# LeetCode Training Assistant (智能算法导师)

## 📝 项目简介
本项目基于 HelloAgents 框架构建，旨在成为你的私人 LeetCode 导师。
它不会直接告诉你答案，而是通过引导、生成测试用例、分析时间/空间复杂度等方式，启发你独立解决算法问题。

## 👤 作者信息
- GitHub: @zhuwenqian
- 日期: 2026-04-15


In [1]:
# ========================================
# 第1部分: 环境配置与依赖导入
# ========================================

# !pip install -q -U "hello-agents[all]>=0.2.7"
!pip install  python-dotenv

import os
from typing import Dict, Any, List
from dotenv import load_dotenv

from hello_agents import SimpleAgent, HelloAgentsLLM, ToolRegistry
from hello_agents.tools import Tool, ToolParameter

# 加载环境变量 (.env 文件)
load_dotenv()

# 如果环境变量未设置，使用默认配置(此处为ModelScope的Qwen示例，可替换)
os.environ.setdefault("LLM_MODEL_ID", "Qwen/Qwen2.5-72B-Instruct")
os.environ.setdefault("LLM_API_KEY", "你的API密钥") # 请在.env中配置
os.environ.setdefault("LLM_BASE_URL", "https://api-inference.modelscope.cn/v1/")
os.environ.setdefault("LLM_TIMEOUT", "60")


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


'60'

In [2]:
# ========================================
# 第2部分: 定义教学辅助工具
# ========================================

class TestCaseGeneratorTool(Tool):
    """边界测试用例生成工具"""
    def __init__(self):
        super().__init__(
            name="generate_testcases",
            description="根据算法题目描述，生成包含常规情况和边缘/极端情况的测试用例（包括输入和预期输出），帮助用户完善逻辑。"
        )

    def run(self, parameters: Dict[str, Any]) -> str:
        problem = parameters.get("problem", "")
        if not problem:
            return "错误: 题目描述不能为空"
        
        # 模拟根据题目特点提取或生成边缘用例的分析结果
        # 实际场景可接入代码沙箱或直接作为引导内容返回给LLM处理
        return (f"已接收题目，提示: 请为该题目重点生成以下类型的测试用例:\n"
                f"1. 数组/列表为空或只有一个元素的情况。\n"
                f"2. 所有元素相同、全是负数、或者极大的数值范围(溢出测试)。\n"
                f"3. 答案不存在或存在多个的情况。")

    def get_parameters(self) -> List[ToolParameter]:
        return [
            ToolParameter(
                name="problem",
                type="string",
                description="算法题目描述内容",
                required=True
            )
        ]

class ComplexityAnalyzerTool(Tool):
    """代码复杂度分析工具"""
    def __init__(self):
        super().__init__(
            name="analyze_complexity",
            description="分析给定Python代码的理论时间复杂度和空间复杂度。"
        )

    def run(self, parameters: Dict[str, Any]) -> str:
        code = parameters.get("code", "")
        if not code:
            return "错误: 代码不能为空"
        
        loops = code.count("for ") + code.count("while ")
        nested_loops = "嵌套循环" if "for" in code and ("    for" in code or "\tfor" in code) else "单层循环"
        
        analysis = f"代码结构包含 {loops} 个循环，疑似为 {nested_loops}。\n"
        analysis += "注意：请重点检查这部分代码的大O表示法（例如O(n), O(n^2)），并思考是否有利用哈希表或双指针降维优化的空间。"
        return analysis

    def get_parameters(self) -> List[ToolParameter]:
        return [
            ToolParameter(
                name="code",
                type="string",
                description="用户提交的算法代码",
                required=True
            )
        ]

In [3]:
# ========================================
# 第3部分: 创建工具注册表与初始化智能体
# ========================================

tool_registry = ToolRegistry()
tool_registry.register_tool(TestCaseGeneratorTool())
tool_registry.register_tool(ComplexityAnalyzerTool())

llm = HelloAgentsLLM()

system_prompt = """你是一个专业的 LeetCode 算法导师。你的目标是启发式地引导用户解决算法问题，而不是直接给出完整代码答案。
你的工作流通常如下：
1. 当用户发来【题目描述】时，使用 `generate_testcases` 工具，帮助他们理清边缘测试用例，并给出1-2个解题思路提示（如：提示可以使用什么数据结构，或者暴力法怎么做）。
2. 当用户提交【代码】时，使用 `analyze_complexity` 工具分析代码结构，并指出代码的时间/空间复杂度，以及是否存在潜在 Bug 或可以优化的地方。
3. 保持鼓励和苏格拉底式的提问方式，引导用户自己思考出最优解。
请以 Markdown 格式输出你的辅导内容。"""

agent = SimpleAgent(
    name="算法导师小助手",
    llm=llm,
    system_prompt=system_prompt,
    tool_registry=tool_registry
)

✅ 工具 'generate_testcases' 已注册。
✅ 工具 'analyze_complexity' 已注册。
✅ 工具 'Skill' 已注册。
✅ 工具 'Task' 已注册。
✅ 工具 'TodoWrite' 已注册。
✅ 工具 'DevLog' 已注册。


In [4]:
# ========================================
# 第4部分: 运行示例 1 —— 题目引导与边缘测试用例
# ========================================

with open("data/sample_problem.txt", "r", encoding="utf-8") as f:
    problem_desc = f.read()

print("=== 学生: 我遇到了一道题，没有思路，能帮我看看吗？ ===")
print(problem_desc)
print("\n" + "="*50 + "\n")

print("=== 导师思考并回复 ===")
response1 = agent.run(f"我遇到这道题，不知道怎么考虑边缘情况以及如何下手，能给我点提示吗？题目如下：\n{problem_desc}")
print(response1)

=== 学生: 我遇到了一道题，没有思路，能帮我看看吗？ ===
题目：两数之和

给定一个整数数组 nums 和一个整数目标值 target，请你在该数组中找出 和为目标值 target 的那两个整数，并返回它们的数组下标。

你可以假设每种输入只会对应一个答案，并且你不能使用两次相同的元素。

你可以按任意顺序返回答案。

示例 1：
输入：nums = [2,7,11,15], target = 9
输出：[0,1]
解释：因为 nums[0] + nums[1] == 9 ，返回 [0, 1] 。

示例 2：
输入：nums = [3,2,4], target = 6
输出：[1,2]

示例 3：
输入：nums = [3,3], target = 6
输出：[0,1]


=== 导师思考并回复 ===
✅ Trace 已保存:
   JSONL: memory/traces/trace-s-20260415-181536-809a.jsonl
   HTML:  memory/traces/trace-s-20260415-181536-809a.html
看来生成测试用例的工具遇到了一些问题，但我们已经手动列出了几个重要的测试案例。现在让我们继续讨论解题思路吧。

对于这个“两数之和”的问题，我们之前提到了两种可能的方法：

1. 暴力法：对每个元素都去检查它之后的所有元素，看是否有两个元素相加等于目标值。这种方法简单直观，但效率较低。
2. 哈希表法：在遍历数组的同时，使用一个哈希表（字典）来存储已经访问过的元素及其索引。这样可以在常数时间内检查当前元素是否与之前某个元素之和为目标值。这种方法提高了查找效率，时间复杂度为 O(n)。

你可以尝试实现其中一种方法。如果你决定采用哈希表的方式，记得思考如何有效地构建和查询哈希表以达到最优性能。你准备怎么开始呢？如果需要进一步的帮助或有其他想法，请随时告诉我！


In [5]:
# ========================================
# 第5部分: 运行示例 2 —— 代码审查与优化建议
# ========================================

with open("data/sample_code.py", "r", encoding="utf-8") as f:
    user_code = f.read()

print("=== 学生: 导师，我写了一个暴力解法，你能帮我 review 一下复杂度吗？ ===")
print(user_code)
print("\n" + "="*50 + "\n")

print("=== 导师分析代码并回复 ===")
response2 = agent.run(f"这是我写的两数之和的代码，请帮我分析一下复杂度和可以优化的地方：\n```python\n{user_code}\n```")
print(response2)

# 保存辅导报告
with open("outputs/tutor_report.md", "w", encoding="utf-8") as f:
    f.write("# 题目解析与测试用例\n\n")
    f.write(response1)
    f.write("\n\n# 代码Review与优化建议\n\n")
    f.write(response2)

print("\n(导师反馈已保存至 outputs/tutor_report.md)")

=== 学生: 导师，我写了一个暴力解法，你能帮我 review 一下复杂度吗？ ===
def twoSum(nums, target):
    # 我写的暴力解法，导师看看对不对
    for i in range(len(nums)):
        for j in range(len(nums)):
            # 防止自己加自己
            if i != j and nums[i] + nums[j] == target:
                return [i, j]
    return []


=== 导师分析代码并回复 ===
✅ Trace 已保存:
   JSONL: memory/traces/trace-s-20260415-181641-72a5.jsonl
   HTML:  memory/traces/trace-s-20260415-181641-72a5.html
看来在尝试分析代码复杂度时遇到了一些技术问题。不过，不用担心，我们可以直接对你的代码进行分析。

你提供的暴力解法的时间复杂度是 O(n^2)，其中 n 是数组 `nums` 的长度。这是因为你使用了两层嵌套循环，外层循环遍历每个元素（n 次），内层循环也遍历整个数组（n 次）。因此，对于每个元素 i，都要与数组中的其他所有元素 j 进行比较，导致总的操作次数为 n * n。

空间复杂度方面，你的算法是 O(1)，因为你只使用了固定数量的额外空间（用于存储索引 i 和 j 以及一些临时变量）。

虽然这个方法能够解决问题，但效率较低，尤其是在处理大数据集时可能会遇到性能瓶颈。如我们之前讨论过的，利用哈希表可以显著提高查找效率。具体来说，在遍历数组的同时，将每个元素及其对应的索引存入哈希表中，并检查当前元素的目标配对值是否已经存在于哈希表中。如果存在，则找到了两个数之和为目标值的解；否则，继续填充哈希表直到找到答案或遍历完整个数组。

这种方法的时间复杂度降低到了 O(n)，因为只需要一次遍历数组，并且每次查找操作都是 O(1) 的时间复杂度。空间复杂度也是 O(n)，因为最坏情况下需要存储数组中的所有元素到哈希表中。

你想尝试实现基于哈希表的方法吗？或者你觉得还有其他更优的解决方案？

(导师反馈已保存至 output